In [ ]:
# Build knowledge base from code smell definitions
smells_match = "smells/content/smells/**/*.md"
all_smells = glob(smells_match, recursive=True)

headers_to_split_on = [("#", "Title"), ("##", "Section"), ("###", "Subsection")]
docs = []

for file in all_smells:
    with open(file, 'r', encoding='utf-8') as f:
        content = f.read()
    if '---' in content:
        parts = content.split('---', 2)
        markdown_content = '--' + parts[2] if len(parts) >= 3 else content
    else:
        markdown_content = content
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    header_splits = markdown_splitter.split_text(markdown_content)
    for doc in header_splits:
        doc.metadata['source'] = file
    docs.extend(header_splits)

print(f"✅ Processed {len(all_smells)} code smell definition files into {len(docs)} document chunks")

# Create embeddings and vector store
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
dataset_path = 'mem://deeplake/smells'
smell_db = DeepLake.from_documents(docs, embeddings, dataset_path=dataset_path)
retriever = smell_db.as_retriever()
retriever.search_kwargs['distance_metric'] = 'cos'
retriever.search_kwargs['k'] = 20

print("✅ Knowledge base created with DeepLake vector store")

In [ ]:
# Clone repositories for code smell definitions and test data
!git clone https://github.com/watabou/pixel-dungeon.git
!git clone https://github.com/Luzkan/smells.git

# SonarQube Code Analysis with Weave Evaluation and Database Integration

This notebook demonstrates a complete pipeline for:
1. **SonarQube API Integration**: Fetch analysis results from your local SonarQube instance
2. **Wandb Weave Tracking**: Log experiments and create datasets from DataFrames
3. **Google Drive Export**: Store results as CSV files in Google Drive
4. **MySQL Database**: Set up and query MySQL database in Google Colab
5. **F1-Score Evaluation**: Track model performance with comprehensive metrics

## Prerequisites
- Local SonarQube instance running with FastAPI bridge
- Wandb account and API key
- Google Drive API credentials
- MySQL database access

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q -U wandb weave litellm "langchain-google-genai" "deeplake<4.0.0" "langchain" "langchain-text-splitters" "langchain-community" "tiktoken" "google-ai-generativelanguage==0.6.15" python-dotenv python-frontmatter
!pip install -q pandas numpy scikit-learn requests
!pip install -q mysql-connector-python PyDrive2 google-auth google-auth-oauthlib google-auth-httplib2
!pip install -q pydantic fastapi uvicorn python-multipart

In [ ]:
# Import required libraries
import os
import json
import pandas as pd
import numpy as np
import requests
import mysql.connector
import litellm
from glob import glob
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
from pydantic import BaseModel, Field
import enum

# LangChain and DeepLake for RAG pipeline
from langchain.output_parsers import PydanticOutputParser
from langchain.vectorstores import DeepLake
from langchain.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.schema import Document
from IPython.display import Markdown, display

# ML and evaluation
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Wandb and Weave
import wandb
import weave
from weave import Dataset, Evaluation

# Google Drive integration
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth, drive
from oauth2client.client import GoogleCredentials

print("✅ All libraries imported successfully")

## 2. Configuration and Authentication

In [ ]:
# Configuration
SONARQUBE_API_URL = "http://your-local-ip:8000"  # Your FastAPI bridge URL
WANDB_PROJECT = "smellai-sonarqube-evaluation"
WANDB_ENTITY = "havrp-org"  # Replace with your wandb entity

# Environment setup for Colab
if 'google.colab' in str(get_ipython()):
    # Mount Google Drive
    drive.mount('/content/drive')
    
    # Authenticate for Google Drive API
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    google_drive = GoogleDrive(gauth)
    
    print("✅ Google Colab environment configured")
else:
    print("ℹ️  Running in local environment")

In [ ]:
# Wandb and Weave initialization
# Set your API keys here or use environment variables
WANDB_API_KEY = "your_wandb_api_key"  # Replace with your key
GOOGLE_API_KEY = "your_google_api_key"  # Replace with your Google API key
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Initialize Wandb
wandb.login()
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    notes="SonarQube analysis with Weave evaluation pipeline and LLM integration",
    tags=["sonarqube", "code-analysis", "f1-score", "weave", "llm", "rag"]
)

# Initialize Weave
weave.init(f"{WANDB_ENTITY}/{WANDB_PROJECT}")

print("✅ Wandb and Weave initialized")

## 3. MySQL Database Setup for Google Colab

In [ ]:
# Install and configure MySQL in Colab
if 'google.colab' in str(get_ipython()):
    # Install MySQL server
    !apt-get -y install mysql-server
    
    # Start MySQL service
    !service mysql start
    
    # Set root password and secure installation
    !mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH 'mysql_native_password' BY 'root';FLUSH PRIVILEGES;"
    
    print("✅ MySQL installed and configured in Colab")
else:
    print("ℹ️  Using existing MySQL installation")

In [ ]:
# Database configuration
DB_CONFIG = {
    'user': 'root',
    'password': 'root',
    'host': 'localhost',
    'database': 'smellai_analysis'
}

def setup_database():
    """Create database and tables for storing analysis results"""
    try:
        # Connect without specifying database first
        conn = mysql.connector.connect(
            user=DB_CONFIG['user'],
            password=DB_CONFIG['password'],
            host=DB_CONFIG['host']
        )
        cursor = conn.cursor()
        
        # Create database
        cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_CONFIG['database']}")
        cursor.execute(f"USE {DB_CONFIG['database']}")
        
        # Create tables
        cursor.execute('''
        CREATE TABLE IF NOT EXISTS sonarqube_projects (
            id INT AUTO_INCREMENT PRIMARY KEY,
            project_key VARCHAR(255) NOT NULL UNIQUE,
            project_name VARCHAR(255),
            last_analysis_date DATETIME,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        ''')
        
        cursor.execute('''
        CREATE TABLE IF NOT EXISTS code_issues (
            id INT AUTO_INCREMENT PRIMARY KEY,
            project_key VARCHAR(255),
            issue_key VARCHAR(255) UNIQUE,
            rule VARCHAR(255),
            severity VARCHAR(50),
            component VARCHAR(500),
            line_number INT,
            message TEXT,
            issue_type VARCHAR(100),
            status VARCHAR(50),
            creation_date DATETIME,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (project_key) REFERENCES sonarqube_projects(project_key)
        )
        ''')
        
        cursor.execute('''
        CREATE TABLE IF NOT EXISTS evaluation_results (
            id INT AUTO_INCREMENT PRIMARY KEY,
            project_key VARCHAR(255),
            model_name VARCHAR(255),
            f1_score FLOAT,
            precision_score FLOAT,
            recall_score FLOAT,
            accuracy FLOAT,
            total_predictions INT,
            correct_predictions INT,
            evaluation_date DATETIME,
            wandb_run_id VARCHAR(255),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        ''')
        
        conn.commit()
        cursor.close()
        conn.close()
        
        print("✅ Database and tables created successfully")
        return True
        
    except Exception as e:
        print(f"❌ Database setup failed: {e}")
        return False

# Setup database
setup_database()

## 3.5. Building the Knowledge Base with RAG Pipeline

We create a RAG pipeline to provide the LLM with context about code smells. The knowledge base is built from a repository of code smell definitions and stored in a DeepLake vector store.

## 4. SonarQube API Client Integration

In [ ]:
class SonarQubeClient:
    """Client for interacting with SonarQube FastAPI bridge"""
    
    def __init__(self, api_url: str):
        self.api_url = api_url.rstrip('/')
        self.session = requests.Session()
    
    def configure_sonarqube(self, sonar_url: str, sonar_token: str):
        """Configure SonarQube connection via API"""
        try:
            response = self.session.post(
                f"{self.api_url}/configure",
                json={"url": sonar_url, "token": sonar_token}
            )
            response.raise_for_status()
            print("✅ SonarQube configured successfully")
            return True
        except Exception as e:
            print(f"❌ SonarQube configuration failed: {e}")
            return False
    
    def get_projects(self) -> List[Dict]:
        """Get all projects from SonarQube"""
        try:
            response = self.session.get(f"{self.api_url}/projects")
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"❌ Failed to fetch projects: {e}")
            return []
    
    def get_project_analysis(self, project_key: str, include_issues: bool = True) -> Dict:
        """Get complete analysis for a project"""
        try:
            params = {"include_issues": include_issues}
            response = self.session.get(
                f"{self.api_url}/projects/{project_key}/analysis",
                params=params
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"❌ Failed to fetch analysis for {project_key}: {e}")
            return {}
    
    def get_project_issues(self, project_key: str, severity: str = None) -> List[Dict]:
        """Get issues for a specific project"""
        try:
            params = {}
            if severity:
                params["severity"] = severity
                
            response = self.session.get(
                f"{self.api_url}/projects/{project_key}/issues",
                params=params
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"❌ Failed to fetch issues for {project_key}: {e}")
            return []

# Initialize SonarQube client
sonar_client = SonarQubeClient(SONARQUBE_API_URL)

# Configure SonarQube connection (replace with your details)
SONAR_URL = "http://localhost:9000"  # Your SonarQube URL
SONAR_TOKEN = "your_sonarqube_token"  # Your SonarQube token

sonar_client.configure_sonarqube(SONAR_URL, SONAR_TOKEN)

## 5. Data Collection and DataFrame Creation

In [ ]:
@weave.op()
def collect_sonarqube_data(client: SonarQubeClient) -> pd.DataFrame:
    """Collect SonarQube data and create a DataFrame"""
    all_data = []
    
    # Get all projects
    projects = client.get_projects()
    print(f"Found {len(projects)} projects")
    
    for project in projects:
        project_key = project['key']
        print(f"Collecting data for project: {project_key}")
        
        # Get project analysis
        analysis = client.get_project_analysis(project_key)
        
        if not analysis:
            continue
            
        # Process issues
        for issue in analysis.get('issues', []):
            row = {
                'project_key': project_key,
                'project_name': project.get('name', ''),
                'issue_key': issue.get('key', ''),
                'rule': issue.get('rule', ''),
                'severity': issue.get('severity', ''),
                'component': issue.get('component', ''),
                'line': issue.get('line'),
                'message': issue.get('message', ''),
                'type': issue.get('type', ''),
                'status': issue.get('status', ''),
                'creation_date': issue.get('creationDate', ''),
                'collection_timestamp': datetime.now().isoformat()
            }
            all_data.append(row)
        
        # Add metrics as separate rows or combine with issues
        metrics = analysis.get('metrics', [])
        for metric in metrics:
            row = {
                'project_key': project_key,
                'project_name': project.get('name', ''),
                'metric_name': metric.get('metric', ''),
                'metric_value': metric.get('value', ''),
                'collection_timestamp': datetime.now().isoformat(),
                'data_type': 'metric'
            }
            # Store metrics separately or combine based on your needs
    
    df = pd.DataFrame(all_data)
    print(f"✅ Collected {len(df)} records")
    return df

# Collect data
sonarqube_df = collect_sonarqube_data(sonar_client)
print(f"DataFrame shape: {sonarqube_df.shape}")
sonarqube_df.head()

## 6. Weave Dataset Creation from DataFrame

In [ ]:
# Create Weave Dataset from DataFrame
def create_weave_dataset_from_df(df: pd.DataFrame, dataset_name: str) -> Dataset:
    """Create a Weave Dataset from a pandas DataFrame"""
    try:
        # Convert DataFrame to Weave Dataset
        dataset = Dataset.from_pandas(df, name=dataset_name)
        
        # Publish the dataset to Weave
        weave.publish(dataset)
        
        print(f"✅ Created Weave dataset '{dataset_name}' with {len(df)} rows")
        return dataset
        
    except Exception as e:
        print(f"❌ Failed to create Weave dataset: {e}")
        return None

# Create dataset
dataset_name = f"sonarqube-analysis-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
weave_dataset = create_weave_dataset_from_df(sonarqube_df, dataset_name)

# Log dataset info to Wandb
wandb.log({
    "dataset_name": dataset_name,
    "dataset_size": len(sonarqube_df),
    "unique_projects": sonarqube_df['project_key'].nunique() if 'project_key' in sonarqube_df.columns else 0,
    "collection_timestamp": datetime.now().isoformat()
})

## 7. Google Drive CSV Export

In [ ]:
def export_to_google_drive(df: pd.DataFrame, filename: str, drive_folder_id: str = None):
    """Export DataFrame to Google Drive as CSV"""
    try:
        # Save to local CSV first
        local_path = f"/content/{filename}"
        df.to_csv(local_path, index=False)
        
        # Upload to Google Drive
        if 'google.colab' in str(get_ipython()):
            file_metadata = {
                'title': filename,
                'mimeType': 'text/csv'
            }
            
            if drive_folder_id:
                file_metadata['parents'] = [{'id': drive_folder_id}]
            
            drive_file = google_drive.CreateFile(file_metadata)
            drive_file.SetContentFile(local_path)
            drive_file.Upload()
            
            print(f"✅ Exported {filename} to Google Drive (ID: {drive_file['id']})")
            return drive_file['id']
        else:
            # For local environment, just save to local directory
            local_save_path = f"./exports/{filename}"
            os.makedirs("./exports", exist_ok=True)
            df.to_csv(local_save_path, index=False)
            print(f"✅ Exported {filename} to {local_save_path}")
            return local_save_path
            
    except Exception as e:
        print(f"❌ Export failed: {e}")
        return None

# Export main dataset
export_filename = f"sonarqube_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
drive_file_id = export_to_google_drive(sonarqube_df, export_filename)

# Log export info
wandb.log({
    "exported_file": export_filename,
    "drive_file_id": drive_file_id,
    "export_timestamp": datetime.now().isoformat()
})

# Define LLM-based evaluation models and schemas following your pattern
class CodeSmellSeverity(str, enum.Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"

class CodeSmellDetection(BaseModel):
    smell_type: str = Field(description="The type of code smell detected")
    location: str = Field(description="Where in the code the smell was found")
    severity: CodeSmellSeverity = Field(description="How severe the smell is")
    description: str = Field(description="Brief explanation of the issue")
    refactoring_suggestion: str = Field(description="How to fix the code smell")

class CodeAnalysisResult(BaseModel):
    analysis_summary: str = Field(description="Overall summary of code quality")
    smells_detected: List[CodeSmellDetection] = Field(description="List of detected code smells")

class SonarQubeCodeAnalysisModel(weave.Model):
    """LLM-based code analysis model that integrates SonarQube data with RAG pipeline"""
    llm_model: str
    temperature: float
    use_sonarqube_context: bool = True

    @weave.op()
    def predict(self, code_content: str, sonarqube_issues: List[Dict] = None) -> Optional[CodeAnalysisResult]:
        """Analyze code using LLM with SonarQube context and knowledge base"""
        
        # Create dynamic code smell enum from knowledge base
        CodeSmell = enum.Enum('CodeSmell', {p.stem.replace('-', '_').upper(): p.stem for p in
                                            Path('smells/content/smells').rglob('*.md')})
        valid_smells = [smell.name for smell in CodeSmell]

        parser = PydanticOutputParser(pydantic_object=CodeAnalysisResult)
        
        # Get relevant context from knowledge base
        retrieved_docs = retriever.get_relevant_documents(code_content)
        retrieved_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
        
        # Prepare SonarQube context if available
        sonarqube_context = ""
        if self.use_sonarqube_context and sonarqube_issues:
            sonarqube_context = "\n**SonarQube Issues Context:**\n"
            for issue in sonarqube_issues[:10]:  # Limit to avoid token overflow
                sonarqube_context += f"- {issue.get('rule', 'Unknown')}: {issue.get('message', 'No message')} (Severity: {issue.get('severity', 'Unknown')})\n"

        prompt = f"""
You are an expert code analyst. Analyze the following code for code smells based on the provided context.

**Context on Code Smells:**
{retrieved_context}

{sonarqube_context}

**Code to Analyze:**
```java
{code_content}
```

**Instructions:**
Only identify code smells from this list: {valid_smells}.
Consider both the knowledge base context and SonarQube issues (if provided) to make accurate assessments.
Focus on providing actionable refactoring suggestions.

Format your response as a structured JSON object matching this schema:
{parser.get_format_instructions()}
"""

        try:
            response = litellm.completion(
                model=self.llm_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=self.temperature,
                response_format={"type": "json_object"}
            )
            response_content = response.choices[0].message.content
            if isinstance(response_content, str):
                response_json = json.loads(response_content)
            else:
                response_json = response_content
            return CodeAnalysisResult.parse_obj(response_json)
        except Exception as e:
            print(f"Error during LLM code analysis: {e}")
            return None

    @weave.op()
    def predict_from_sonarqube_data(self, sonarqube_row: Dict) -> Optional[CodeAnalysisResult]:
        """Predict code smells from SonarQube data row"""
        # Extract relevant information from SonarQube data
        component = sonarqube_row.get('component', '')
        
        # Try to read the actual file if component path is available
        code_content = ""
        if component and os.path.exists(component):
            try:
                with open(component, 'r', encoding='utf-8') as f:
                    code_content = f.read()
            except:
                # Fallback to using issue message as context
                code_content = f"File: {component}\nIssue: {sonarqube_row.get('message', '')}"
        else:
            # Use available SonarQube information as context
            code_content = f"""
            Component: {component}
            Rule: {sonarqube_row.get('rule', '')}
            Message: {sonarqube_row.get('message', '')}
            Type: {sonarqube_row.get('type', '')}
            """
        
        # Create issues context from current row
        issues_context = [sonarqube_row] if sonarqube_row else None
        
        return self.predict(code_content, issues_context)

# Initialize models with different configurations
baseline_model = SonarQubeCodeAnalysisModel(
    llm_model="gemini/gemini-2.0-flash-exp", 
    temperature=0.1,
    use_sonarqube_context=False
)

enhanced_model = SonarQubeCodeAnalysisModel(
    llm_model="gemini/gemini-2.0-flash-exp", 
    temperature=0.1,
    use_sonarqube_context=True
)

print("✅ LLM-based evaluation framework initialized with RAG pipeline")

In [ ]:
@weave.op()
def evaluate_llm_model_performance(df: pd.DataFrame, model: SonarQubeCodeAnalysisModel) -> EvaluationMetrics:
    """Evaluate LLM model performance and calculate F1-score"""
    
    predictions = []
    ground_truth = []
    successful_predictions = 0
    
    print(f"Evaluating {model.model_name} on {len(df)} samples...")
    
    for idx, (_, row) in enumerate(df.iterrows()):
        if pd.notna(row.get('issue_key')):
            try:
                # Get LLM prediction
                analysis_result = model.predict_from_sonarqube_data(row.to_dict())
                
                if analysis_result and analysis_result.smells_detected:
                    # Use the first detected smell for evaluation
                    first_smell = analysis_result.smells_detected[0]
                    predicted_severity = first_smell.severity.value
                    predictions.append(predicted_severity)
                    
                    # Map SonarQube severity to our severity scale
                    actual_severity = row.get('severity', 'MINOR')
                    severity_mapping = {
                        'BLOCKER': 'HIGH',
                        'CRITICAL': 'HIGH', 
                        'MAJOR': 'MEDIUM',
                        'MINOR': 'LOW',
                        'INFO': 'LOW'
                    }
                    mapped_severity = severity_mapping.get(actual_severity, 'LOW')
                    ground_truth.append(mapped_severity)
                    
                    successful_predictions += 1
                else:
                    # If no smells detected, predict LOW severity
                    predictions.append('LOW')
                    actual_severity = row.get('severity', 'MINOR')
                    severity_mapping = {
                        'BLOCKER': 'HIGH',
                        'CRITICAL': 'HIGH', 
                        'MAJOR': 'MEDIUM',
                        'MINOR': 'LOW',
                        'INFO': 'LOW'
                    }
                    mapped_severity = severity_mapping.get(actual_severity, 'LOW')
                    ground_truth.append(mapped_severity)
                    
            except Exception as e:
                print(f"Error processing row {idx}: {e}")
                # Skip this row
                continue
                
        if idx > 0 and idx % 10 == 0:
            print(f"Processed {idx} samples, {successful_predictions} successful predictions")
    
    if not predictions or not ground_truth:
        return EvaluationMetrics(
            f1_score=0.0, precision=0.0, recall=0.0, accuracy=0.0,
            support=0, model_name=model.model_name,
            evaluation_timestamp=datetime.now().isoformat()
        )
    
    # Calculate metrics
    f1 = f1_score(ground_truth, predictions, average='weighted')
    precision = precision_score(ground_truth, predictions, average='weighted')
    recall = recall_score(ground_truth, predictions, average='weighted')
    accuracy = sum(p == g for p, g in zip(predictions, ground_truth)) / len(predictions)
    
    # Log detailed classification report
    report = classification_report(ground_truth, predictions, output_dict=True)
    wandb.log({
        f"{model.model_name}_classification_report": report,
        f"{model.model_name}_successful_predictions": successful_predictions,
        f"{model.model_name}_total_attempts": len(df)
    })
    
    metrics = EvaluationMetrics(
        f1_score=f1,
        precision=precision,
        recall=recall,
        accuracy=accuracy,
        support=len(predictions),
        model_name=model.model_name,
        evaluation_timestamp=datetime.now().isoformat()
    )
    
    return metrics

# Evaluate both models
print("🔄 Starting model evaluation...")

# Limit evaluation dataset for demo (remove .head(50) for full evaluation)
eval_df = sonarqube_df.head(50) if not sonarqube_df.empty else pd.DataFrame()

if not eval_df.empty:
    # Evaluate baseline model (without SonarQube context)
    baseline_results = evaluate_llm_model_performance(eval_df, baseline_model)
    print(f"\\n📊 Baseline Model Results:")
    print(f"F1-Score: {baseline_results.f1_score:.4f}")
    print(f"Precision: {baseline_results.precision:.4f}")
    print(f"Recall: {baseline_results.recall:.4f}")
    print(f"Accuracy: {baseline_results.accuracy:.4f}")
    
    # Evaluate enhanced model (with SonarQube context)
    enhanced_results = evaluate_llm_model_performance(eval_df, enhanced_model)
    print(f"\\n📊 Enhanced Model Results:")
    print(f"F1-Score: {enhanced_results.f1_score:.4f}")
    print(f"Precision: {enhanced_results.precision:.4f}")
    print(f"Recall: {enhanced_results.recall:.4f}")
    print(f"Accuracy: {enhanced_results.accuracy:.4f}")
    
    # Log comparison
    wandb.log({
        "model_comparison": {
            "baseline_f1": baseline_results.f1_score,
            "enhanced_f1": enhanced_results.f1_score,
            "improvement": enhanced_results.f1_score - baseline_results.f1_score
        }
    })
else:
    print("⚠️  No data available for evaluation")

## 9. F1-Score Evaluation Framework

In [ ]:
@weave.op()
def code_smell_accuracy_scorer(issue_data: Dict, model_output: CodeAnalysisResult) -> Dict:
    """Score the accuracy of code smell detection"""
    if not model_output or not model_output.smells_detected:
        return {
            "correct": False,
            "detected_smells": 0,
            "actual_severity": issue_data.get('severity', 'UNKNOWN'),
            "predicted_severity": "NONE",
            "reason": "No smells detected"
        }
    
    # Map SonarQube severity to our scale
    severity_mapping = {
        'BLOCKER': 'HIGH',
        'CRITICAL': 'HIGH', 
        'MAJOR': 'MEDIUM',
        'MINOR': 'LOW',
        'INFO': 'LOW'
    }
    
    actual_severity = issue_data.get('severity', 'MINOR')
    mapped_actual = severity_mapping.get(actual_severity, 'LOW')
    
    # Check if any detected smell matches the severity
    severity_match = False
    predicted_severities = []
    
    for smell in model_output.smells_detected:
        predicted_severities.append(smell.severity.value)
        if smell.severity.value == mapped_actual:
            severity_match = True
            break
    
    return {
        "correct": severity_match,
        "detected_smells": len(model_output.smells_detected),
        "actual_severity": mapped_actual,
        "predicted_severities": predicted_severities,
        "smell_types": [smell.smell_type for smell in model_output.smells_detected]
    }

@weave.op()
def analysis_quality_scorer(issue_data: Dict, model_output: CodeAnalysisResult) -> Dict:
    """Score the quality of the analysis (presence of meaningful suggestions)"""
    if not model_output:
        return {"quality_score": 0.0, "has_suggestions": False, "analysis_length": 0}
    
    quality_score = 0.0
    
    # Check if analysis summary exists and is meaningful
    if model_output.analysis_summary and len(model_output.analysis_summary) > 20:
        quality_score += 0.3
    
    # Check if smells were detected
    if model_output.smells_detected:
        quality_score += 0.4
        
        # Check if refactoring suggestions are provided
        has_meaningful_suggestions = any(
            smell.refactoring_suggestion and len(smell.refactoring_suggestion) > 10
            for smell in model_output.smells_detected
        )
        if has_meaningful_suggestions:
            quality_score += 0.3
    
    return {
        "quality_score": quality_score,
        "has_suggestions": bool(model_output.smells_detected),
        "analysis_length": len(model_output.analysis_summary) if model_output.analysis_summary else 0,
        "detected_smell_count": len(model_output.smells_detected) if model_output.smells_detected else 0
    }

# Create evaluation dataset from SonarQube data
eval_rows = []
for _, row in sonarqube_df.head(20).iterrows():  # Limit for demo
    if pd.notna(row.get('issue_key')):
        eval_rows.append(row.to_dict())

if eval_rows:
    print(f"Creating Weave evaluation with {len(eval_rows)} samples...")
    
    evaluation_dataset = Dataset(
        name=f"sonarqube-llm-eval-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
        rows=eval_rows
    )
    
    # Publish the evaluation dataset
    weave.publish(evaluation_dataset)
    
    # Create and run evaluation for both models
    evaluation = Evaluation(
        name="SonarQube LLM Analysis Evaluation",
        dataset=evaluation_dataset,
        scorers=[code_smell_accuracy_scorer, analysis_quality_scorer]
    )
    
    print("🔄 Running Weave evaluation for baseline model...")
    baseline_eval_results = await evaluation.evaluate(baseline_model)
    
    print("🔄 Running Weave evaluation for enhanced model...")
    enhanced_eval_results = await evaluation.evaluate(enhanced_model)
    
    print("✅ Weave evaluations completed")
    print(f"Baseline evaluation: {baseline_eval_results}")
    print(f"Enhanced evaluation: {enhanced_eval_results}")
    
    # Log evaluation results to Wandb
    wandb.log({
        "weave_baseline_results": baseline_eval_results,
        "weave_enhanced_results": enhanced_eval_results
    })
    
else:
    print("⚠️  No valid evaluation data found for Weave evaluation")

In [ ]:
@weave.op()
def evaluate_model_performance(df: pd.DataFrame, model: CodeSmellAnalysisModel) -> EvaluationMetrics:
    """Evaluate model performance and calculate F1-score"""
    
    predictions = []
    ground_truth = []
    
    for _, row in df.iterrows():
        if pd.notna(row.get('issue_key')):
            # Get prediction
            prediction = model.predict(row.to_dict())
            predictions.append(prediction.predicted_severity.value)
            
            # Use actual severity as ground truth
            actual_severity = row.get('severity', 'MINOR')
            ground_truth.append(actual_severity)
    
    if not predictions or not ground_truth:
        return EvaluationMetrics(
            f1_score=0.0, precision=0.0, recall=0.0, accuracy=0.0,
            support=0, model_name=model.model_name,
            evaluation_timestamp=datetime.now().isoformat()
        )
    
    # Calculate metrics
    f1 = f1_score(ground_truth, predictions, average='weighted')
    precision = precision_score(ground_truth, predictions, average='weighted')
    recall = recall_score(ground_truth, predictions, average='weighted')
    accuracy = sum(p == g for p, g in zip(predictions, ground_truth)) / len(predictions)
    
    metrics = EvaluationMetrics(
        f1_score=f1,
        precision=precision,
        recall=recall,
        accuracy=accuracy,
        support=len(predictions),
        model_name=model.model_name,
        evaluation_timestamp=datetime.now().isoformat()
    )
    
    return metrics

# Run evaluation
evaluation_results = evaluate_model_performance(sonarqube_df, baseline_model)

print(f"\n📊 Evaluation Results for {evaluation_results.model_name}:")
print(f"F1-Score: {evaluation_results.f1_score:.4f}")
print(f"Precision: {evaluation_results.precision:.4f}")
print(f"Recall: {evaluation_results.recall:.4f}")
print(f"Accuracy: {evaluation_results.accuracy:.4f}")
print(f"Support: {evaluation_results.support}")

def store_evaluation_results(metrics: EvaluationMetrics, project_key: str = "all"):
    """Store evaluation results in MySQL and log to Wandb"""
    try:
        # Store in MySQL
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        
        cursor.execute(
            """
            INSERT INTO evaluation_results 
            (project_key, model_name, f1_score, precision_score, recall_score, 
             accuracy, total_predictions, correct_predictions, evaluation_date, wandb_run_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """,
            (
                project_key,
                metrics.model_name,
                metrics.f1_score,
                metrics.precision,
                metrics.recall,
                metrics.accuracy,
                metrics.support,
                int(metrics.accuracy * metrics.support),
                datetime.now(),
                wandb.run.id
            )
        )
        
        conn.commit()
        cursor.close()
        conn.close()
        
        # Log to Wandb
        wandb.log({
            f"{metrics.model_name}_f1_score": metrics.f1_score,
            f"{metrics.model_name}_precision": metrics.precision,
            f"{metrics.model_name}_recall": metrics.recall,
            f"{metrics.model_name}_accuracy": metrics.accuracy,
            f"{metrics.model_name}_support": metrics.support,
            "evaluation_timestamp": metrics.evaluation_timestamp
        })
        
        print(f"✅ Evaluation results for {metrics.model_name} stored successfully")
        return True
        
    except Exception as e:
        print(f"❌ Failed to store evaluation results for {metrics.model_name}: {e}")
        return False

# Store results for both models if they exist
if 'baseline_results' in locals():
    store_evaluation_results(baseline_results)

if 'enhanced_results' in locals():
    store_evaluation_results(enhanced_results)

# Create comprehensive summary DataFrame
summary_data = []

if 'baseline_results' in locals():
    summary_data.append({
        'model_name': baseline_results.model_name,
        'model_type': 'LLM_without_SonarQube_context',
        'f1_score': baseline_results.f1_score,
        'precision': baseline_results.precision,
        'recall': baseline_results.recall,
        'accuracy': baseline_results.accuracy,
        'support': baseline_results.support,
        'evaluation_timestamp': baseline_results.evaluation_timestamp,
        'wandb_run_id': wandb.run.id
    })

if 'enhanced_results' in locals():
    summary_data.append({
        'model_name': enhanced_results.model_name,
        'model_type': 'LLM_with_SonarQube_context',
        'f1_score': enhanced_results.f1_score,
        'precision': enhanced_results.precision,
        'recall': enhanced_results.recall,
        'accuracy': enhanced_results.accuracy,
        'support': enhanced_results.support,
        'evaluation_timestamp': enhanced_results.evaluation_timestamp,
        'wandb_run_id': wandb.run.id
    })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    
    # Export summary to Google Drive
    summary_filename = f"llm_evaluation_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    export_to_google_drive(summary_df, summary_filename)
    
    print("\n📊 LLM Model Comparison Summary:")
    print(summary_df.to_string(index=False))
    
    # Calculate improvement if both models were evaluated
    if len(summary_data) == 2:
        improvement = summary_data[1]['f1_score'] - summary_data[0]['f1_score']
        print(f"\n📈 F1-Score Improvement with SonarQube context: {improvement:.4f}")
        
        wandb.log({
            "final_f1_improvement": improvement,
            "baseline_f1": summary_data[0]['f1_score'],
            "enhanced_f1": summary_data[1]['f1_score']
        })
else:
    print("⚠️  No evaluation results to summarize")

In [ ]:
@weave.op()
def severity_accuracy_scorer(issue_data: Dict, model_output: CodeSmellPrediction) -> Dict:
    """Score the accuracy of severity prediction"""
    actual_severity = issue_data.get('severity', 'MINOR')
    predicted_severity = model_output.predicted_severity.value
    
    correct = actual_severity == predicted_severity
    
    return {
        "correct": correct,
        "actual_severity": actual_severity,
        "predicted_severity": predicted_severity,
        "confidence": model_output.confidence
    }

@weave.op()
def type_accuracy_scorer(issue_data: Dict, model_output: CodeSmellPrediction) -> Dict:
    """Score the accuracy of type prediction"""
    actual_type = issue_data.get('type', 'CODE_SMELL')
    predicted_type = model_output.predicted_type
    
    correct = actual_type == predicted_type
    
    return {
        "correct": correct,
        "actual_type": actual_type,
        "predicted_type": predicted_type
    }

# Create evaluation dataset
eval_rows = []
for _, row in sonarqube_df.head(100).iterrows():  # Limit for demo
    if pd.notna(row.get('issue_key')):
        eval_rows.append(row.to_dict())

if eval_rows:
    evaluation_dataset = Dataset(
        name=f"sonarqube-eval-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
        rows=eval_rows
    )
    
    # Create and run evaluation
    evaluation = Evaluation(
        name="SonarQube Analysis Evaluation",
        dataset=evaluation_dataset,
        scorers=[severity_accuracy_scorer, type_accuracy_scorer]
    )
    
    # Run evaluation
    results = await evaluation.evaluate(baseline_model)
    
    print("✅ Weave evaluation completed")
    print(f"Evaluation results: {results}")
else:
    print("⚠️  No valid evaluation data found")

## 11. Results Storage and Tracking

In [ ]:
def store_evaluation_results(metrics: EvaluationMetrics, project_key: str = "all"):
    """Store evaluation results in MySQL and log to Wandb"""
    try:
        # Store in MySQL
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        
        cursor.execute(
            """
            INSERT INTO evaluation_results 
            (project_key, model_name, f1_score, precision_score, recall_score, 
             accuracy, total_predictions, correct_predictions, evaluation_date, wandb_run_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """,
            (
                project_key,
                metrics.model_name,
                metrics.f1_score,
                metrics.precision,
                metrics.recall,
                metrics.accuracy,
                metrics.support,
                int(metrics.accuracy * metrics.support),
                datetime.now(),
                wandb.run.id
            )
        )
        
        conn.commit()
        cursor.close()
        conn.close()
        
        # Log to Wandb
        wandb.log({
            f"{metrics.model_name}_f1_score": metrics.f1_score,
            f"{metrics.model_name}_precision": metrics.precision,
            f"{metrics.model_name}_recall": metrics.recall,
            f"{metrics.model_name}_accuracy": metrics.accuracy,
            f"{metrics.model_name}_support": metrics.support,
            "evaluation_timestamp": metrics.evaluation_timestamp
        })
        
        print("✅ Evaluation results stored successfully")
        return True
        
    except Exception as e:
        print(f"❌ Failed to store evaluation results: {e}")
        return False

# Store results
store_evaluation_results(evaluation_results)

# Create summary DataFrame for export
summary_df = pd.DataFrame([{
    'model_name': evaluation_results.model_name,
    'f1_score': evaluation_results.f1_score,
    'precision': evaluation_results.precision,
    'recall': evaluation_results.recall,
    'accuracy': evaluation_results.accuracy,
    'support': evaluation_results.support,
    'evaluation_timestamp': evaluation_results.evaluation_timestamp,
    'wandb_run_id': wandb.run.id
}])

# Export summary to Google Drive
summary_filename = f"evaluation_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
export_to_google_drive(summary_df, summary_filename)

print("\n📊 Final Summary:")
print(summary_df.to_string(index=False))

## 12. Query and Analysis Functions

In [ ]:
def query_evaluation_history() -> pd.DataFrame:
    """Query evaluation history from MySQL"""
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        
        query = """
        SELECT 
            model_name,
            f1_score,
            precision_score,
            recall_score,
            accuracy,
            total_predictions,
            evaluation_date,
            wandb_run_id
        FROM evaluation_results 
        ORDER BY evaluation_date DESC
        """
        
        df = pd.read_sql(query, conn)
        conn.close()
        
        return df
        
    except Exception as e:
        print(f"❌ Query failed: {e}")
        return pd.DataFrame()

def analyze_project_issues(project_key: str) -> Dict:
    """Analyze issues for a specific project"""
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        
        query = """
        SELECT 
            severity,
            issue_type,
            COUNT(*) as issue_count
        FROM code_issues 
        WHERE project_key = %s
        GROUP BY severity, issue_type
        ORDER BY issue_count DESC
        """
        
        df = pd.read_sql(query, conn, params=[project_key])
        conn.close()
        
        return df.to_dict('records')
        
    except Exception as e:
        print(f"❌ Analysis failed: {e}")
        return {}

# Query evaluation history
history_df = query_evaluation_history()
if not history_df.empty:
    print("\n📈 Evaluation History:")
    print(history_df.to_string(index=False))
else:
    print("\n📈 No evaluation history found")

# Analyze issues for the first project (if available)
if not sonarqube_df.empty and 'project_key' in sonarqube_df.columns:
    first_project = sonarqube_df['project_key'].iloc[0]
    project_analysis = analyze_project_issues(first_project)
    if project_analysis:
        print(f"\n🔍 Issue Analysis for {first_project}:")
        for item in project_analysis[:5]:  # Show top 5
            print(f"  {item['severity']} {item['issue_type']}: {item['issue_count']} issues")

## 13. Cleanup and Summary

In [ ]:
# Create final summary
final_summary = {
    "experiment_id": wandb.run.id,
    "total_projects_analyzed": sonarqube_df['project_key'].nunique() if 'project_key' in sonarqube_df.columns else 0,
    "total_issues_collected": len(sonarqube_df),
    "dataset_name": dataset_name,
    "exported_files": [export_filename, summary_filename],
    "model_performance": {
        "f1_score": evaluation_results.f1_score,
        "precision": evaluation_results.precision,
        "recall": evaluation_results.recall,
        "accuracy": evaluation_results.accuracy
    },
    "completion_timestamp": datetime.now().isoformat()
}

# Log final summary
wandb.log({"final_summary": final_summary})

# Print summary
print("\n🎉 Pipeline Execution Complete!")
print("\n📋 Summary:")
for key, value in final_summary.items():
    if key != "model_performance":
        print(f"  {key}: {value}")
    else:
        print(f"  Model Performance:")
        for metric, score in value.items():
            print(f"    {metric}: {score:.4f}")

# Finish Wandb run
wandb.finish()

print("\n✅ All tasks completed successfully!")
print("📊 Check your Wandb dashboard for detailed tracking")
print("💾 Data stored in MySQL database and exported to Google Drive")
print("🔍 Weave datasets and evaluations are available for further analysis")